# 05 — Metric Signature Emergence

## Claim

The thesis claims that:
1. The metric signature $(-,+,+,+)$ is **axiomatic** in GR, not derived from anything
2. It is "doubly Cartesian": both the 4D manifold format and the binary sign split belong to the measurement framework
3. A description built from **nested cycles** would not generate a metric signature at all
4. From inside a system of four nested concentric oscillators, an observer would encounter
   a gradient from "coordinate identification succeeds" (inner, returnable) to "coordinate
   identification fails" (outer, non-returnable) — and the Cartesian parser would code this
   gradient as a binary $(+)$ vs. $(-)$ distinction
5. The sign split is the Cartesian observer's parsing of a continuous gradient, not an intrinsic binary

This is the most ambitious notebook. We test whether nested oscillators, observed from inside,
produce the **apparent** asymmetry between returnable and non-returnable coordinates.

## Model

1. Build a system of $N$ nested oscillators
2. Define an "internal observer" who measures coordinate return probability
3. Compute the return probability as a function of nesting depth
4. Show that this forms a gradient, not a binary
5. Show that a threshold-based observer converts this gradient into a binary sign

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product
import seaborn as sns

sns.set_theme(style='whitegrid')
%matplotlib inline

### 1. Coordinate Identification Success Rate

For each orbit in a nested system, ask: if I observe the system at two times when
this orbit has returned to (approximately) the same angular position, how often
has the **total state** also returned to within $\epsilon$?

- Inner orbits: inner content is simple → state often matches → coordinate identification **succeeds**
- Outer orbits: inner content is complex → state rarely matches → coordinate identification **fails**

This is the **operational definition** of "returnable vs. non-returnable."

In [ ]:
# Four nested oscillators with prime frequencies
primes = np.array([2, 3, 5, 7])
omega = 2 * np.pi * primes

# Long time series
N_points = 1_000_000
t = np.linspace(0, 100, N_points)

# Angular positions
theta = np.array([np.mod(omega[i] * t, 2 * np.pi) for i in range(4)])

def coordinate_identification_rate(orbit_idx, theta_all, epsilon_position=0.1, epsilon_state=0.1):
    """
    For orbits at index orbit_idx:
    1. Find times when this orbit's angular position is within epsilon_position of t=0
    2. Among those times, count how often the FULL state (all orbits) is within epsilon_state of t=0
    3. Return the success rate = coordinate identification rate
    """
    # Times when this orbit returns to initial position
    initial_pos = theta_all[orbit_idx, 0]
    pos_diff = np.abs(theta_all[orbit_idx] - initial_pos)
    pos_diff = np.minimum(pos_diff, 2*np.pi - pos_diff)  # circular
    position_returns = pos_diff < epsilon_position
    position_returns[0] = False  # exclude t=0
    
    if position_returns.sum() == 0:
        return 0.0, 0
    
    # Among those times, check full state distance from initial
    return_indices = np.where(position_returns)[0]
    initial_state = theta_all[:, 0]
    
    state_matches = 0
    for idx in return_indices:
        state = theta_all[:, idx]
        diff = np.abs(state - initial_state)
        diff = np.minimum(diff, 2*np.pi - diff)
        dist = np.linalg.norm(diff)
        if dist < epsilon_state:
            state_matches += 1
    
    rate = state_matches / len(return_indices)
    return rate, len(return_indices)

# Compute for each orbit
print('Coordinate Identification Success Rate:')
print('=' * 65)
print(f'{"Orbit":>8} | {"Position returns":>16} | {"State matches":>14} | {"Rate":>8}')
print('-' * 65)

rates = []
for i, p in enumerate(primes):
    rate, n_returns = coordinate_identification_rate(i, theta, 
                                                     epsilon_position=0.15,
                                                     epsilon_state=0.3)
    rates.append(rate)
    print(f'{f"Orbit {p}":>8} | {n_returns:>16} | {int(rate*n_returns):>14} | {rate:>8.4f}')

print()
print('→ Inner orbits: high success rate (coordinate identification works)')
print('→ Outer orbits: low success rate (coordinate identification fails)')
print('→ This is a GRADIENT, not a binary.')

In [ ]:
# Visualise the gradient
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']
ax1.bar([f'Orbit {p}' for p in primes], rates, color=colors)
ax1.set_ylabel('Coordinate identification success rate', fontsize=11)
ax1.set_title('Success Rate by Nesting Level\n(gradient from returnable to non-returnable)', fontsize=12)
ax1.set_ylim(0, 1)

# Now show: thresholding this gradient creates the binary +/- sign
threshold = 0.5  # arbitrary threshold
signs = ['+' if r > threshold else '−' for r in rates]
sign_colors = ['#4CAF50' if r > threshold else '#E91E63' for r in rates]

ax2.bar([f'Orbit {p}' for p in primes], rates, color=sign_colors)
ax2.axhline(y=threshold, color='black', linestyle='--', linewidth=2, label=f'Threshold = {threshold}')
ax2.set_ylabel('Success rate → sign assignment', fontsize=11)
ax2.set_title('Binary Threshold Creates (+,+,+,−) from Gradient\n(the Cartesian parser at work)', fontsize=12)
ax2.set_ylim(0, 1)

# Add sign labels
for i, (p, r, s) in enumerate(zip(primes, rates, signs)):
    ax2.text(i, r + 0.03, s, ha='center', va='bottom', fontsize=20, fontweight='bold')

ax2.legend(fontsize=10)

fig.suptitle('From Continuous Gradient to Binary Signature:\nHow the Cartesian Observer Generates $(-,+,+,+)$', fontsize=13, y=1.04)
plt.tight_layout()
plt.show()

### 2. Sensitivity to Threshold

The gradient is continuous. Different thresholds produce different binary splits.
If the gradient always crosses from high to low between orbits 3 and 4 (primes 5 and 7),
then $3+1$ is stable regardless of where the threshold is placed — a robust property of the nesting.

But if the gradient shifts with parameters, the $3+1$ split is an artifact of where
the observer places the threshold.

In [ ]:
# Compute identification rates at multiple epsilon values
epsilons = [0.05, 0.1, 0.2, 0.3, 0.5]

print('Coordinate identification rate vs. precision (ε_state):')
print('=' * 70)
header = f'{"Orbit":>8}'
for e in epsilons:
    header += f' | {f"ε={e}":>8}'
print(header)
print('-' * 70)

all_rates = {}
for e in epsilons:
    orbit_rates = []
    for i, p in enumerate(primes):
        rate, _ = coordinate_identification_rate(i, theta,
                                                 epsilon_position=0.15, 
                                                 epsilon_state=e)
        orbit_rates.append(rate)
    all_rates[e] = orbit_rates

for i, p in enumerate(primes):
    row = f'{f"Orbit {p}":>8}'
    for e in epsilons:
        row += f' | {all_rates[e][i]:>8.4f}'
    print(row)

# Check: where does the gradient drop?
print()
print('Gradient drop location (orbit where rate drops below 50%):')
for e in epsilons:
    drop = None
    for i, r in enumerate(all_rates[e]):
        if r < 0.5:
            drop = primes[i]
            break
    print(f'  ε = {e}: drops at orbit {drop if drop else "(never)"}')

In [ ]:
# Visualise all epsilon curves
fig, ax = plt.subplots(figsize=(10, 6))

for e in epsilons:
    ax.plot(range(4), all_rates[e], 'o-', label=f'ε = {e}', linewidth=2, markersize=8)

ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
ax.set_xticks(range(4))
ax.set_xticklabels([f'Orbit {p}\n(prime {p})' for p in primes])
ax.set_ylabel('Coordinate identification success rate', fontsize=11)
ax.set_xlabel('Nesting level (innermost → outermost)', fontsize=11)
ax.set_title('The Gradient Across Nesting Levels\n(continuous, not binary — binary split depends on threshold)', fontsize=12)
ax.legend(title='State precision ε')
ax.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.show()

### 3. The Phase Space Volume Argument

An alternative approach: compute the **phase space volume** that must be revisited
for full state recurrence at each nesting level.

For $k$ nested oscillators with frequencies $p_1, \ldots, p_k$, the phase space is a $k$-torus.
The fraction of phase space within $\epsilon$ of a point scales as $\epsilon^k / V_k$.
Recurrence probability in time $T$ is proportional to $T \cdot \epsilon^k / V_k$.

In [ ]:
# Phase space analysis
# For a k-torus with side 2π, volume = (2π)^k
# ε-ball volume ≈ ε^k · V_k(1) where V_k(1) is the unit ball volume in k-D
# V_k(1) = π^(k/2) / Γ(k/2 + 1)

from scipy.special import gamma

def unit_ball_volume(k):
    return np.pi**(k/2) / gamma(k/2 + 1)

epsilon = 0.1
print('Phase space recurrence probability (relative):')
print('=' * 65)
print(f'{"Depth k":>8} | {"Torus volume":>14} | {"ε-ball volume":>14} | {"Fraction":>14}')
print('-' * 65)

fractions = []
for k in range(1, 8):
    torus_vol = (2 * np.pi) ** k
    ball_vol = epsilon**k * unit_ball_volume(k)
    frac = ball_vol / torus_vol
    fractions.append(frac)
    print(f'{k:>8} | {torus_vol:>14.2f} | {ball_vol:>14.2e} | {frac:>14.2e}')

print()
print('→ Recurrence probability drops EXPONENTIALLY with nesting depth.')
print('→ The gradient is continuous — no intrinsic binary split.')
print('→ A threshold-based observer creates the split wherever their')
print('   resolution limit falls.')

In [ ]:
# Final visualisation: the gradient and the Cartesian parser
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

k_range = np.arange(1, 8)
log_fractions = np.log10(fractions)

ax1.plot(k_range, log_fractions, 's-', color='steelblue', linewidth=2, markersize=10)
ax1.set_xlabel('Nesting depth k', fontsize=12)
ax1.set_ylabel('log₁₀(recurrence probability)', fontsize=12)
ax1.set_title('Recurrence Probability vs. Nesting Depth\n(continuous exponential decay)', fontsize=12)
ax1.set_xticks(k_range)

# The Cartesian parser: impose binary threshold
threshold_k = 3.5  # between k=3 and k=4
ax2.barh(k_range, -np.array(log_fractions), 
         color=['#4CAF50' if k <= 3 else '#E91E63' for k in k_range])
ax2.axvline(x=-log_fractions[2] + 0.5, color='black', linestyle='--', linewidth=2)
ax2.set_ylabel('Nesting depth k', fontsize=12)
ax2.set_xlabel('−log₁₀(recurrence probability)', fontsize=12)
ax2.set_title('The Cartesian Parser:\nImpose threshold → 3 "returnable" + 1 "non-returnable"', fontsize=12)
ax2.set_yticks(k_range)

# Add +/- labels  
for k in k_range:
    sign = '+' if k <= 3 else '−'
    ax2.text(-log_fractions[k-1] + 0.3, k, sign, 
             fontsize=16, fontweight='bold', va='center', color='white')

fig.suptitle('The Metric Signature Is the Cartesian Parser\'s Output\n'
             'The nesting produces a gradient; the parser produces $(−,+,+,+)$', 
             fontsize=13, y=1.06)
plt.tight_layout()
plt.show()

### 4. Summary: What the Nesting Produces vs. What the Parser Reports

| Level | What the nesting produces | What the Cartesian parser reports |
|-------|--------------------------|----------------------------------|
| Orbit 2 (innermost) | Fast recurrence, simple state | Coordinate 1: spatial (+) |
| Orbit 3 | Moderate recurrence, moderate state complexity | Coordinate 2: spatial (+) |
| Orbit 5 | Slow recurrence, complex state | Coordinate 3: spatial (+) |
| Orbit 7 (outermost) | Astronomical recurrence, state complexity = entire inner system | Coordinate 4: temporal (−) |

The gradient is **real**. The binary split is the **parser's output**.

The metric signature $(-,+,+,+)$ is not a property of the nesting. It is what the Cartesian
observer's measurement framework produces when applied to the nesting.

## Verdict

*(To be filled after running the computation)*